In [1]:
import ants

In [10]:
target = ants.image_read(
    "/scratch/imaging/ds005237/sub-NDARINVXP963GZ5/anat/sub-NDARINVXP963GZ5_run-01_T1w.nii.gz"
)
donor = ants.image_read(
    "/work/charite/neuroimaging/lea/Pilot3_02_07_2026_LeaW/_mprage_sag_0p1mm_20260702103322_6.nii"
)
mask = target.new_image_like(target.numpy() != 0.0)  # pyright: ignore[reportAttributeAccessIssue]

In [13]:
registration = ants.registration(
    fixed=target,
    moving=donor,
    type_of_transform="antsRegistrationSyN[b]",
    mask=mask,
    mask_all_stages=True,
    verbose=True,
)

antsRegistration --dimensionality 3 -r [0x72aaf6db9128,0x72ac44107048,1] --output [/tmp/tmpl4vd1697,0x72aaf6db92a8,0x72aaf6db92c8] --transform Rigid[0.1] --metric MI[0x72aaf6db9128,0x72ac44107048,1,32,Regular,0.25] --convergence [1000x500x250x100,1e-6,10] --shrink-factors 8x4x2x1 --smoothing-sigmas 3x2x1x0vox --transform Affine[0.1] --metric MI[0x72aaf6db9128,0x72ac44107048,1,32,Regular,0.25] --convergence [1000x500x250x100,1e-6,10] --shrink-factors 8x4x2x1 --smoothing-sigmas 3x2x1x0vox --transform BSplineSyN[0.2,26,0,3] --metric CC[0x72aaf6db9128,0x72ac44107048,1,2] --convergence [100x70x50x20,1e-6,10] --shrink-factors 8x4x2x1 --smoothing-sigmas 3x2x1x0vox -x [0x72aaf6db9288,NA] --float 1 --write-composite-transform 0 -v 1
All_Command_lines_OK
Using single precision for computations.
The composite transform comprises the following transforms (in order): 
  1. Center of mass alignment using fixed image: 0x72aaf6db9128 and moving image: 0x72ac44107048 (type = Euler3DTransform)
  Reading

In [15]:
padded_target = ants.pad_image(target, pad_width=[(60, 60)] * 3)

warped_donor = ants.apply_transforms(
    fixed=padded_target,
    moving=donor,
    transformlist=registration["fwdtransforms"],
)

# trim the empty border (crop_image keeps the physical coordinates correct)
data = warped_donor.numpy()  # pyright: ignore[reportAttributeAccessIssue, reportOptionalMemberAccess]
mask = data > 0.02 * data.max()
warped_donor = ants.crop_image(
    warped_donor,
    warped_donor.new_image_like(mask.astype("float32")),  # pyright: ignore[reportOptionalMemberAccess, reportAttributeAccessIssue]
)


In [16]:
# warped_donor = registration["warpedmovout"]
ants.image_write(warped_donor, "t1w.nii.gz")